
### Attention Mechanism implemented from scratch using only NumPy.
Includes:
- softmax
- scaled dot-product attention (with optional masking)
- multi-head attention
- a small runnable demo

Reference: "Attention Is All You Need" (Vaswani et al., 2017)


In [ ]:
gpu_available = torch.cuda.is_available()
print(f"GPU Available: {gpu_available}")

if gpu_available:
    # Get the number of available GPUs
    print(f"GPU Count: {torch.cuda.device_count()}")
    
    # Get the ID of the current GPU
    print(f"Current GPU ID: {torch.cuda.current_device()}")
    
    # Get the name of the GPU
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


In [ ]:
### Dependency
import torch
import torch.nn as nn 
import numpy as np 

In [ ]:
###. Softmax function 
def softmax_function(x, axis= -1):
    x = x - np.max(x, axis = axis,  keepdims = True)
    return np.exp(x)/ np.sum(np.exp(x), axis = axis, keepdims = True)

# softmax_function(x=1, axis = -1)

In [ ]:
### Scaled dot product_attention 
""" Q: (..., seq_len_q, d_k)
    K: (..., seq_len_k, d_k)
    V: (..., seq_len_k, d_v)
    mask: (..., seq_len_q, seq_len_k), 0 where masked out, 1 where allowed
          (or None for no masking)"""

def scaled_dot_product_attention(Q, K,  V,  mask = None):
    d_k = Q.shape[-1] 

    similarity_score = Q @ np.swapaxes(K,-1, -2)

    similarity_score = similarity_score / np.sqrt(d_k)

    if mask is not None: 
        similarity_score = np.where(mask == 0, -1e9, similarity_score)
    # This lets you block certain query-key pairs from attending to each other. Wherever mask == 0, the score gets replaced with a huge negative number (-1e9). Why? Because after softmax, e^(-1e9) is essentially 0 — so those positions get effectively zero attention weight.
    attention_weight = softmax_function(similarity_score, axis = -1)
    # for every Q, the attention weight across every K equavalent to 1

    result_attention = attention_weight @ V

    return result_attention, attention_weight

In [ ]:
### Multi-Head attention 
class MultiHeadAttention:
    def __init__(self, d_model, numb_heads, seed = 42):
        assert d_model % numb_heads == 0 
        self.d_k = d_model // numb_heads
        self.d_model = d_model
        self.numb_heads = numb_heads
        rang = np.random.default_rng(seed)

        def init(shape):
            limit = np.sqrt(6 / (shape[0] + shape[-1]))
            return rang.uniform(-limit, limit, size = shape)
        self.W_q = init((d_model, d_model))
        self.W_k = init((d_model, d_model))
        self.W_v = init((d_model, d_model))
        self.W_o = init((d_model, d_model))

    def split_heads(self, x):
        batch, seq_len, _ = x.shape
        x = x.reshape(batch, seq_len, self.numb_heads, self.d_k)
        return x.tranpose(0, 2, 1 ,3)

    def combine_heads(self, x):
        batch, self.numb_heads, seq_len, d_k = x.shape
        x = x.tranpose(0, 2, 1, 3)
        return x.reshape(batch, seq_len, self.numb_heads * d_k)

    def __call__(self, x_q, x_k, x_v, mask = None):
        Q = x_q @ self.W_q
        K = x_k @ self.W_k
        V = x_v @ self.W_v

        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        if mask is not None: 
            mask = mask[:, None, :, :]

        attention_output, attention_weigths = scaled_dot_product_attention(Q, K,  V, mask)

        attention_output = self.combine_heads(attention_output)
        output = attention_output @ self.W_o

        return output, attention_weigths

def causal_mask(seq_len):
    return np.tril(np.ones((seq_len, seq_len)))


if __name__ == "__main__":
    np.set_printoptions(precision = 3, suppress = True)

    batch, seq_len, d_model, numb_heads = 2, 5, 16, 4


    rang = np.random.default_rng(42)


    x = rang.normal(size = (batch, seq_len, d_model))

    mha = MultiHeadAttention(d_model, numb_heads, seed =1)


    # Self-attention, no mask
    out, weights = mha(x, x, x)
    print("Self-attention output shape:", out.shape)         # (2, 5, 16)
    print("Attention weights shape:   ", weights.shape)       # (2, 4, 5, 5)
    print("Row sums (should be ~1):\n", weights[0, 0].sum(axis=-1))

    # Causal self-attention (e.g. for a decoder / language model)
    mask = np.stack([causal_mask(seq_len)] * batch)  # (batch, seq_len, seq_len)
    out_causal, weights_causal = mha(x, x, x, mask=mask)
    print("\nCausal attention weights for batch 0, head 0:")
    print(weights_causal[0,0])




### PyTorch Implementation 


In [ ]:
import torch
import torch.nn as nn

embed_dim = 256
num_heads = 8
seq_len = 50
batch_size = 32

# Initialize layer
attention_layer = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

# Generate a mock sequence matrix [Batch, Sequence Length, Embedding Dim]
x = torch.randn(batch_size, seq_len, embed_dim)

# Pass identical elements for Q, K, and V to achieve self-attention
attn_output, attn_weights = attention_layer(x, x, x)
print(attn_output.shape) # Output: torch.Size([32, 50, 256])


In [9]:
import torch 
import torch.nn as nn 


embed_dim = 256 
num_heads = 8
batch_size = 32
seq_len = 50 

x = torch.randn(batch_size, seq_len, embed_dim)

attention_layer = torch.nn.MultiheadAttention(embed_dim= embed_dim, num_heads= num_heads, batch_first = True)

atten_output, atten_weights = attention_layer(x,x,x)

print(atten_output.shape) 




torch.Size([32, 50, 256])


In [ ]:
import torch
import torch.nn as nn 
import math

embed_dim = 256 
num_heads = 8
batch_size = 32
seq_len = 50

x = torch.randn(batch_size, seq_len, embed_dim)

attention_layer = torch.nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)

optimizer = torch.optim.Adam(attention_layer.parameters(), lr = 1e-3)


warmup_lr = 4000
total_lr = 100000
base_lr = 1e-3
min_lr = 1e-5

def lr_lambda(step):
    step = max(step, 1) ### automatically reset to zero to avoid the "divide by zero" scenario 

    if step < warmup_lr:
        return step / warmup_lr

    else:
        progress = (step - warmup_lr) / max(1,total_lr - warmup_lr)
        progress = min(progress, 1)
        cosine_term = 0.5 * (1 + math.cos(math.pi * progress))
        return (min_lr + (base_lr - min_lr) * cosine_term) / base_lr 


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


for step in range(total_lr):
    optimizer.zero_grad

    atten_output, atten_weigths = attention_layer(x,x,x)


    loss = atten_output.sum()

    loss.backward()

    optimizer.step()
    scheduler.step()

    if step % 500 == 0:
        current_lr = scheduler.get_last_lr()[0]
        print(f"Step: {step}, LR: {current_lr:.6f}, Loss: {loss.item():4f}")

